# Lab 7 — Connect the five stages
### EP0418 · Topic 8 · 2 practical hours

---
You have built every piece separately. Today you connect them — and the joins are
where systems actually break.

## The method: skeleton first

**Do not** connect five finished components and hope. Build the whole chain with
everything faked, get it running end to end, then replace one stage at a time.

If you connect five untested things and it fails, you have five suspects.

## Putting the assistant's code into a TODO cell

A TODO cell has a line ending in `...`. Replace **only that line** with the
assistant's code, and keep the name on the left (`s`, `clean`, `H`). Delete any
`import` or `read_csv` lines it adds; the data is already loaded. If it used a
different name for the result, add one line at the end to rename it, e.g.
`s = hourly_df`.

---
## How every lab in this module works

You will **not** write this code from scratch. Ask your AI assistant to write it.
Your job is the part it cannot do: deciding what the right answer looks like,
reading what it gave you, and proving it did what you asked.

At the foot of each task is a **verification contract** — a few `check.*` lines.
**You may not edit them.** The AI can write any implementation it likes; it cannot
satisfy these by accident.

If a check fails, read the message. It tells you what was found and what to try
next. Failing checks are the normal state of a lab in progress, not a sign you
are behind.

## Set up Gemini for this notebook (once, 30 seconds)

Open Gemini (the spark icon at the bottom of the notebook). In its chat box, click the **sliders icon** next to **+**, choose **Configure**, paste the text below and save. Leave the mode on **Default**. Colab keeps the instructions with this notebook, so every answer follows the course rules.

```
I am new to coding. Explain any code you write in plain words: what it does to my data.
Before writing code for a bigger task, tell me your plan in plain words.
Never delete or drop rows unless I ask. If a step would remove rows, say so first.
After changing the data, show the row count before and after, and the smallest and largest values.
Codes such as -999 mean a missing reading. Point them out; do not treat them as real numbers.
Do not change any line that starts with check. Those are the lab's locked checks.
If you are not sure, say so. Do not guess.
```

In [ ]:
import os, urllib.request
REPO = "https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/"


def fetch(*names):
    """Download any course file that is not beside the notebook yet."""
    for name in names:
        if os.path.exists(name):
            continue
        try:
            urllib.request.urlretrieve(REPO + name, name)
            print("fetched", name)
        except Exception:
            print(f"Could not download {name}. Upload it with the folder icon on the left.")
            from google.colab import files
            files.upload()


fetch("ep0418.py")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from ep0418 import check


def filled(*values):
    """True once every TODO slot above has been replaced. Otherwise says so."""
    if any(v is ... or v is None for v in values):
        print("A TODO above is still '...'. Fill it in, then run this cell again.")
        return False
    return True

## Task 1 — The skeleton, entirely faked

Get this running in the first twenty minutes. It does nothing useful. That is the
point.

In [ ]:
def ingest():
    """Stage 1 — later: read your real data."""
    return pd.DataFrame({"timestamp": pd.date_range("2026-06-01", periods=10, freq="h"),
                         "value": range(10)})

def clean(df):
    """Stage 2 — later: your real cleaning."""
    return df

def analyse(df):
    """Stage 3 — later: your detector or forecast."""
    return df.assign(flag=False)

def decide(df):
    """Stage 4 — later: your rule, debounced and rate-limited."""
    return df[df.flag]

def respond(events):
    """Stage 5 — later: your real notification."""
    print(f"would send {len(events)} alert(s)")
    return [f"alert {i}" for i in range(len(events))]

def pipeline():
    df = ingest()
    print(f"  ingest  -> {len(df):,} rows")
    df = clean(df)
    print(f"  clean   -> {len(df):,} rows")
    df = analyse(df)
    print(f"  analyse -> {len(df):,} rows")
    events = decide(df)
    print(f"  decide  -> {len(events):,} events")
    return respond(events)

sent = pipeline()

### Note the row count printed at every stage

That is not decoration. **"Zero alerts" and "a stage is silently producing
nothing" look identical from outside.** Printing the shape at each stage is how
you tell them apart — in this lab and in your CA3 report.

## Task 2 — Replace one stage at a time

Swap in your real `ingest`. Run the whole pipeline. Still works? Move to `clean`.

**Run the full chain after every single swap.** When it breaks you will know
exactly which change broke it.

### Turning your earlier cells into a function

Your Lab 3 cleaning was written as loose cells on the classroom data. A stage is
the same lines wrapped in `def`, taking a table in and giving a table back:

```python
def clean(df):
    df = df.copy()
    df.loc[df.humidity_pct == -999, "humidity_pct"] = np.nan     # your Lab 3 lines
    df = df.drop_duplicates("timestamp", keep="last").sort_values("timestamp")
    return df
```

Use your own track's file and columns. The starter pack for your track shows how
to load it; the timestamp column and cadence differ by track:

| Track | File | Time column | Cadence |
|---|---|---|---|
| 1 Smart Money | `smart_money_ohlcv.csv` | `Date` | daily |
| 2 Smart Building | `smart_building_sensors.csv` | `timestamp` | 5 min |
| 3 Smart Farm | `sg_rainfall_daily.csv` | `date` | daily |
| 4 Smart City | `sg_taxi_hourly.csv` | `timestamp` | hourly |
| 5 Smart Self | `smart_self_daily.csv` | `date` | daily |

If your track has no `-999` sentinel or no duplicates, `clean()` may be short.
It must still return the table.

### Or start from the track kit

`track_kit.py` (beside this notebook, upload it with `ep0418.py`) has a
ready-made `ingest()` and `clean()` for every track, each handling that file's
documented faults. Copy the pair for your track, **read every line**, then
adapt it. Every kit `ingest()` returns a table with a `timestamp` column, so the
rest of your pipeline can assume that name.

In [ ]:
fetch("track_kit.py")
# Uncomment the line for your track. Then open track_kit.py and read the two
# functions you just imported: you are responsible for what they do.
# from track_kit import ingest_smart_money    as ingest, clean_smart_money    as clean   # Track 1
# from track_kit import ingest_smart_building as ingest, clean_smart_building as clean   # Track 2
# from track_kit import ingest_smart_farm     as ingest, clean_smart_farm     as clean   # Track 3
# from track_kit import ingest_smart_city     as ingest, clean_smart_city     as clean   # Track 4
# from track_kit import ingest_smart_self     as ingest, clean_smart_self     as clean   # Track 5
# from track_kit import ingest_free_form      as ingest, clean_free_form      as clean   # Track 6
# sent = pipeline()

In [ ]:
# TODO(you): replace ingest() with your real data loading, then re-run the whole chain
...
sent = pipeline()

In [ ]:
# TODO(you): replace clean(), then re-run the whole chain
...
sent = pipeline()

In [ ]:
# TODO(you): replace analyse(), then re-run the whole chain
...
sent = pipeline()

In [ ]:
# TODO(you): replace decide() and respond(), then re-run the whole chain
...
sent = pipeline()

## Task 3 — Verify the whole chain

The contract below checks the things that break at the joins.

In [ ]:
raw_df   = ingest()
clean_df = clean(raw_df)

# TODO(you): explain any change in row count between the two stages
EXPLANATION = "TODO(you): explain any change in row count through your pipeline"

# ---- verification contract — do not edit ----
check.rows_accounted_for(
    raw_df, clean_df,
    names=["ingest", "clean"],
    explain=EXPLANATION,
)
check.response_fired(sent, minimum=1)
check.report()

## Task 4 — Draw it

Five boxes, four arrows. On each arrow write what passes between the stages —
**what shape, what cadence, what columns**.

This diagram goes into your CA3 report, so it is not throwaway work.

---
## Read before you run

A student replaced the analyse stage of their skeleton with this. The pipeline still ran end to end without an error:

**Do not run it yet.** Read it, answer the two questions in the cell below,
and only then remove the `#` and run it to see whether you were right.

The skeleton's `ingest()` returns ten rows with `value` 0 to 9. How many rows come out of this stage? Your debounce from Laboratory 6 looks at “the previous three rows”. After this change, which rows are the previous rows?

Write down the rows in and rows out, and why stage 4 can no longer tell three consecutive readings from three scattered ones.

In [ ]:
# --- AI-generated. Read it first. ---
# def analyse(df):
#     return df[df.value > 5].assign(flag=True)

**What is wrong with it:**

**What it would produce, and what it should produce:**

---
## Before you leave

- The whole chain runs end to end from real data to a real notification.
- `verification_log.md`: which join broke first, and what was the mismatched
  assumption?
- A **Decisions** line for what passes between your stages (cadence, columns) and
  why. Before you close the notebook, ask your assistant to summarise what it
  changed today; check the summary against your row counts and correct it before
  you paste it in.